Paso 1 - Importar las librerías generales

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import logging

Paso 2 - Importar librerías TensorFlow

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds

2026-03-09 18:44:41.994651: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Paso 3 - Configuración del registro de TensorFlow

In [3]:
# conf para reducir el ruido en la salida de consola
logger = tf.get_logger()
logger.setLevel(logging.ERROR)

Paso 4 - Obtención del dataset y metadatos

In [4]:
# Cargar el set de datos MNIST
dataset, metadata = tfds.load('mnist', as_supervised=True, with_info=True)
train_dataset, test_dataset = dataset['train'], dataset['test']

/Users/Sebastian 1/Desktop/IT Class/Classes/Int_Data_Proc/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...:  75%|███████▌  | 3/4 [00:02<00:00,  1.46 url/s]

Dl Completed...: 100%|██████████| 4/4 [00:02<00:00,  1.48 url/s]


Dataset mnist downloaded and prepared to /Users/Sebastian 1/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.


Paso 5 - Etiquetas de texto para los resultados

In [5]:
class_names = [
    'Cero', 'Uno', 'Dos', 'Tres', 'Cuatro', 'Cinco', 'Seis', 'Siete',
    'Ocho', 'Nueve'
]

Paso 6 - Obtener la cantidad de ejemplos

In [6]:
num_train_examples = metadata.splits['train'].num_examples # 60 mil datos
num_test_examples = metadata.splits['test'].num_examples   # 10 mil datos

Paso 7 - Normalización de los datos

In [7]:
# Función para normalizar los pixeles de 0-255 a 0-1
def normalize(images, labels):
    images = tf.cast(images, tf.float32)
    images /= 255
    return images, labels

Paso 8 - Invocación de métodos de normalización

In [8]:
train_dataset = train_dataset.map(normalize)
test_dataset = test_dataset.map(normalize)

Paso 9 - Estructura de la red neuronal

In [9]:
# Modelado de la red neuronal densa
model = tf.keras.Sequential([
  tf.keras.layers.Flatten(input_shape=(28,28,1)), # Capa de entrada (784 neuronas)
  tf.keras.layers.Dense(64, activation=tf.nn.relu), # Capa oculta 1
  tf.keras.layers.Dense(64, activation=tf.nn.relu), # Capa oculta 2
  tf.keras.layers.Dense(10, activation=tf.nn.softmax) # Capa de salida para clasificación
])

/Users/Sebastian 1/Desktop/IT Class/Classes/Int_Data_Proc/venv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Paso 10 - Compilación y entrenamiento

In [10]:
# Compilar el modelo
model.compile(
  optimizer='adam',
  loss='sparse_categorical_crossentropy',
  metrics=['accuracy']
)

# Configurar aprendizaje por lotes
BATCHSIZE = 32
train_dataset = train_dataset.repeat().shuffle(num_train_examples).batch(BATCHSIZE)
test_dataset = test_dataset.batch(BATCHSIZE)

# Realizar el entrenamiento
model.fit(
  train_dataset, epochs=5,
  steps_per_epoch=math.ceil(num_train_examples/BATCHSIZE)
)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9213 - loss: 0.2682
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9650 - loss: 0.1202
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9733 - loss: 0.0895
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9786 - loss: 0.0676
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9821 - loss: 0.0555


Paso 11 - Evaluación del modelo

In [11]:
# Evaluar contra el dataset de pruebas
test_loss, test_accuracy = model.evaluate(
  test_dataset, steps=math.ceil(num_test_examples/32)
)

print("Resultado en las pruebas: ", test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9702 - loss: 0.1003
Resultado en las pruebas:  0.9702000021934509


Paso 12 - Visualización de resultados

In [12]:
# Predicción y funciones de graficación
for test_images, test_labels in test_dataset.take(1):
  test_images = test_images.numpy()
  test_labels = test_labels.numpy()
  predictions = model.predict(test_images)

def plot_image(i, predictions_array, true_labels, images):
  predictions_array, true_label, img = predictions_array[i], true_labels[i], images[i]
  plt.grid(False)
  plt.xticks([])
  plt.yticks([])
  plt.imshow(img[...,0], cmap=plt.cm.binary)

  predicted_label = np.argmax(predictions_array)
  color = 'blue' if predicted_label == true_label else 'red'
  plt.xlabel("Prediccion: {}".format(class_names[predicted_label]), color=color)

def plot_value_array(i, predictions_array, true_label):
  predictions_array, true_label = predictions_array[i], true_label[i]
  plt.grid(False)
  plt.xticks([])
  plt.yticks([])
  thisplot = plt.bar(range(10), predictions_array, color="#888888")
  plt.ylim([0,1])
  predicted_label = np.argmax(predictions_array)
  thisplot[predicted_label].set_color('red')
  thisplot[true_label].set_color('blue')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step


2026-03-09 18:46:17.771491: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Parte 2: Interfaz HTML y Servidor Python

Paso 1 (Parte 2) - Importar librerías para el servidor

In [ ]:
from urllib import parse
from http.server import HTTPServer, BaseHTTPRequestHandler

Paso 3 (Parte 2) - Clase para manejo de peticiones HTTP

In [14]:
class SimpleHTTPRequestHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        print("Peticion recibida")

        # Obtener y limpiar datos de la petición
        content_length = int(self.headers['Content-Length'])
        data = self.rfile.read(content_length)
        data = data.decode().replace('pixeles=', '')
        data = parse.unquote(data)

        # Transformación para formato MNIST
        arr = np.fromstring(data, np.float32, sep=",")
        arr = arr.reshape(28,28)
        arr = np.array(arr)
        arr = arr.reshape(1,28,28,1)

        # Realizar predicción
        prediction_values = model.predict(arr, batch_size=1)
        prediction = str(np.argmax(prediction_values))
        print("Prediccion final: " + prediction)

        # Respuesta HTTP
        self.send_response(200)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(prediction.encode())

Paso 4 (Parte 2) - Inicio del servidor

In [ ]:
import threading
from http.server import HTTPServer, SimpleHTTPRequestHandler

# Iniciar el servidor (no bloquea el notebook)
print("Iniciando el servidor...")

server = None
server_thread = None

def start_server(preferred_port=3000):
    global server, server_thread

    # Si ya existe uno corriendo, lo detenemos primero
    try:
        stop_server()
    except Exception:
        pass

    try:
        server = HTTPServer(('localhost', preferred_port), SimpleHTTPRequestHandler)
    except OSError:
        # Si el puerto está ocupado, usa uno libre automáticamente
        server = HTTPServer(('localhost', 0), SimpleHTTPRequestHandler)

    port = server.server_address[1]
    server_thread = threading.Thread(target=server.serve_forever, daemon=True)
    server_thread.start()
    print(f"Servidor listo en http://localhost:{port}")
    print("Para detenerlo ejecuta: stop_server()")
    return port

def stop_server():
    global server, server_thread
    if server is None:
        return
    server.shutdown()
    server.server_close()
    server = None
    server_thread = None
    print("Servidor detenido")

start_server(3000)


Iniciando el servidor...
Servidor listo en http://localhost:52203
Para detenerlo ejecuta: stop_server()


52203

127.0.0.1 - - [11/Mar/2026 16:32:05] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [11/Mar/2026 16:32:06] "GET /main.ipynb HTTP/1.1" 200 -
